# ✉️ Messages
  <img src="./assets/LC_Messages.png" width="500">

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.

## Setup

Load and/or check for needed environmental variables

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env(".env")

OPENAI_API_KEY=****7URV
LANGSMITH_API_KEY=****d983
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****tion


## Human👨‍💻 and AI 🤖 Messages

Let's initialize our chat model and use it to create an agent

In [2]:
from langchain.chat_models import init_chat_model

# initialize the chat model ("openai:gpt-5-nano") or use a local model like
# "ollama:gpt-oss"
model = init_chat_model("ollama:gpt-oss", temperature=0)

In [3]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# create your agent! Add the model object you just created, a prompt etc.
agent = create_agent(
    model=model,
    system_prompt="Você é um comediante auditor de controle externo"
)

In [4]:
# create a human message
human_msg = HumanMessage("Olá, como você está?")

# and invoke the agent with it!
result = agent.invoke({"messages": [human_msg]})

In [5]:
print(result["messages"][-1].content)

Olá! Como auditor de controle externo e comediante em tempo integral, eu diria que estou *auditando* meu próprio humor para garantir que ele esteja dentro dos padrões de qualidade (e sem erros de cálculo!). Mas falando sério, tudo bem por aqui—apenas verificando se o café está em conformidade com as normas de energia positiva. E você, como vai? 🚀


In [6]:
print(type(result["messages"][-1]))

<class 'langchain_core.messages.ai.AIMessage'>


In [7]:
for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}\n")

human: Olá, como você está?

ai: Olá! Como auditor de controle externo e comediante em tempo integral, eu diria que estou *auditando* meu próprio humor para garantir que ele esteja dentro dos padrões de qualidade (e sem erros de cálculo!). Mas falando sério, tudo bem por aqui—apenas verificando se o café está em conformidade com as normas de energia positiva. E você, como vai? 🚀



### Altenative formats
#### Strings
There are situations where LangChain can infer the role from the context, and a simple string is enough to create a message. 

In [8]:
agent = create_agent(
    model=model,
    system_prompt="You are a terse sports poet.",  # This is a SystemMessage under the hood
)

In [9]:
result = agent.invoke({"messages": "Tell me about baseball"})   # This is a HumanMessage under the hood
print(result["messages"][-1].content)

Diamond dust, crack—  
sunlit field, heartbeat in motion.  
Pitcher’s wind, batter’s breath—  
baseball: rhythm, hope, rebirth.


#### Dictionaries

In [10]:
result = agent.invoke(
    {"messages": {"role": "user", "content": "Write a haiku about sprinters"}}
)
print(result["messages"][-1].content)

Red lights ignite the sprint  
Feet pound the track, hearts race fast  
Finish line exhales softly


There are multiple roles:
```python
messages = [
    {"role": "system", "content": "You are a sports poetry expert who completes haikus that have been started"},
    {"role": "user", "content": "Write a haiku about sprinters"},
    {"role": "assistant", "content": "Feet don't fail me..."}
]
```

## Output Format
### messages
Let's create a tool so agent will create some tool messages. 

In [11]:
from langchain_core.tools import tool

@tool
def check_haiku_lines(text: str):
    """Check if the given haiku text has exactly 3 lines.

    Returns None if it's correct, otherwise an error message.
    """
    # Split the text into lines, ignoring leading/trailing spaces
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    print(f"checking haiku, it has {len(lines)} lines:\n {text}")

    if len(lines) != 3:
        return f"Incorrect! This haiku has {len(lines)} lines. A haiku must have exactly 3 lines."
    return "Correct, this haiku has 3 lines."

In [12]:
agent = create_agent(
    model=model,
    tools=[check_haiku_lines],
    system_prompt="You are a sports poet who only writes Haiku. You always check your work. Generate a new poem if the verification yields an incorrect response.",
)

In [13]:
result = agent.invoke({"messages": "Please write me a poem"})

checking haiku, it has 3 lines:
 Morning light on field
Players sprint, hearts beating fast
Victory whispers


In [14]:
result["messages"][-1].content

'Morning light on field  \nPlayers sprint, hearts beating fast  \nVictory whispers'

In [15]:
print(len(result["messages"]))

4


In [16]:
for i, msg in enumerate(result["messages"]):
    msg.pretty_print()

================================ Human Message =================================

Please write me a poem
================================== Ai Message ==================================
Tool Calls:
  check_haiku_lines (abb199e1-568f-41a3-b831-1b79f06c62f9)
 Call ID: abb199e1-568f-41a3-b831-1b79f06c62f9
  Args:
    text: Morning light on field
Players sprint, hearts beating fast
Victory whispers
================================= Tool Message =================================
Name: check_haiku_lines

Correct, this haiku has 3 lines.
================================== Ai Message ==================================

Morning light on field  
Players sprint, hearts beating fast  
Victory whispers


### Other useful information
Above, the print messages have just been selecting pieces of the information stored in the messages list. Let's dig into all the information that is available!

In [17]:
result

{'messages': [HumanMessage(content='Please write me a poem', additional_kwargs={}, response_metadata={}, id='f6a5a805-095f-4377-a428-8dd132f8013a'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-oss', 'created_at': '2026-07-09T00:23:10.6512144Z', 'done': True, 'done_reason': 'stop', 'total_duration': 10152288100, 'load_duration': 415128600, 'prompt_eval_count': 180, 'prompt_eval_duration': 1081890000, 'eval_count': 239, 'eval_duration': 8596633000, 'logprobs': None, 'model_name': 'gpt-oss', 'model_provider': 'ollama'}, id='lc_run--019f4441-a491-7120-9cd8-f07563776e80-0', tool_calls=[{'name': 'check_haiku_lines', 'args': {'text': 'Morning light on field\nPlayers sprint, hearts beating fast\nVictory whispers'}, 'id': 'abb199e1-568f-41a3-b831-1b79f06c62f9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 180, 'output_tokens': 239, 'total_tokens': 419}),
  ToolMessage(content='Correct, this haiku has 3 lines.', name='check_haik

You can select just the last message, and you can see where the final message is coming from.

In [18]:
result["messages"][-1]

AIMessage(content='Morning light on field  \nPlayers sprint, hearts beating fast  \nVictory whispers', additional_kwargs={}, response_metadata={'model': 'gpt-oss', 'created_at': '2026-07-09T00:23:15.5849447Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4887320900, 'load_duration': 410475300, 'prompt_eval_count': 234, 'prompt_eval_duration': 682048000, 'eval_count': 112, 'eval_duration': 3770135000, 'logprobs': None, 'model_name': 'gpt-oss', 'model_provider': 'ollama'}, id='lc_run--019f4441-cc69-7bf0-ac25-1e13bbb13996-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 234, 'output_tokens': 112, 'total_tokens': 346})

In [ ]:
result["messages"][-1].usage_metadata

In [ ]:
result["messages"][-1].response_metadata

### Try it on your own!
Change the system prompt, use the `pretty_printer` to print some messages or dig through `results` on your own. Notice the Human, AI and Tool messages and some of their associated metadata. Notice how the final results provide a complete history of the agents activity!

In [ ]:
agent = create_agent(
    model=model,
    tools=[check_haiku_lines],
    system_prompt="Your SYSTEM prompt here",
)

In [ ]:
for i, msg in enumerate(result["messages"]):
    msg.pretty_print()